# Del texto a un dato

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un programa no puede usar «Categoría: salud, urgencia alta, parece importante»: necesita campos con nombre y tipo. Y un campo bien formado puede ser falso. |
| **Producto** | Un esquema Pydantic, una tabla de avisos estructurados y una decisión sobre qué campos se pueden automatizar. |
| **Recursos** | El mismo Ollama de los cuadernos anteriores. Unos dos minutos de ejecución. Gemini es opcional (dos llamadas). |

El municipio quiere cargar los avisos en una planilla con categoría, urgencia, lugar y fecha. Pedirle al modelo «devolvé un JSON» parece suficiente hasta que un día agrega un cartel antes de la llave y el programa se rompe. Hay dos problemas distintos: que la respuesta **tenga forma** de dato y que el dato **sea verdadero**.

**¿Cómo conseguimos que la respuesta sea un dato que un programa pueda leer, y cómo sabemos que además es cierto?**

> **Cómo se trabaja.** En clase se presenta cada tema con el código ya escrito. Después lo explorás por tu cuenta: ejecutás las celdas, leés las notas **Para mirar** y cambiás valores para ver qué pasa. Al final de cada bloque hay un **microlaboratorio**: una tabla te dice qué líneas completar y con qué, y la celda siguiente comprueba si quedó bien.

## Los datos

| Campo | Dato |
|---|---|
| Fuente | `clasificacion.json` y `aviso_ner.txt` de la unidad 010: textos sintéticos, CC0-1.0 |
| Unidad de análisis | Un aviso breve con etiqueta (18 para evaluar, de `val` y `test`) y un aviso largo de una actividad cultural |
| Limitaciones | El aviso largo es uno solo y tiene los datos bien ordenados. Los campos `urgencia` y `resumen` no tienen respuesta correcta en los datos: nadie los etiquetó. |

In [1]:
import json
import os

import ollama
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

import utils

utils.comprobar_modelo_local()
utils.versiones(["pydantic", "ollama", "pandas"])

Ollama responde en http://localhost:11434 y tiene granite4:latest.


{'python': '3.12.13',
 'pydantic': '2.14.1',
 'ollama': '0.6.3',
 'pandas': '3.1.0'}

In [2]:
registros = utils.cargar_json("clasificacion.json")
avisos_de_evaluacion = []
for registro in registros:
    if registro["split"] != "train":
        avisos_de_evaluacion.append(registro)

aviso_largo = utils.cargar_texto("aviso_ner.txt")
print(f"Avisos breves: {len(avisos_de_evaluacion)}")
print(f"Aviso largo: {len(aviso_largo.split())} palabras")

Avisos breves: 18
Aviso largo: 62 palabras


## 1. Pedir un JSON con palabras

La forma más directa es pedirlo en el texto: «devolvé un JSON con estas claves». Probamos con los 18 avisos y tratamos de leer cada respuesta con `json.loads`, que convierte un texto JSON en un diccionario de Python.

In [3]:
pedido_json = (
    "Clasificás avisos de vecinos. Devolvé un JSON con dos claves: "
    "categoria (educación, movilidad o salud) y urgencia (un número de 1 a 5)."
)

respuestas_en_texto = []
for registro in avisos_de_evaluacion:
    mensajes = [
        {"role": "system", "content": pedido_json},
        {"role": "user", "content": f"Aviso: {registro['texto']}"},
    ]
    resultado = utils.preguntar(mensajes, maximo_tokens=100)
    respuestas_en_texto.append(resultado["texto"])

print(respuestas_en_texto[0])

```json
{
    "categoria": "educación",
    "urgencia": 3
}
```


In [4]:
leidas = 0
for texto in respuestas_en_texto:
    try:
        json.loads(texto)
        leidas += 1
    except json.JSONDecodeError:
        pass

total = len(respuestas_en_texto)
print(f"Respuestas que json.loads pudo leer: {leidas} de {total}")

Respuestas que json.loads pudo leer: 15 de 18


> **Para mirar.** 15 de 18 respuestas se pudieron leer. Las otras tres vinieron envueltas en un bloque de código (`` ```json ``), que `json.loads` no entiende. Con el mismo pedido y el mismo modelo, la forma cambia de un aviso a otro: por eso un programa no puede suponer que la respuesta va a tener siempre el mismo aspecto.

El arreglo habitual es quitar las marcas del bloque de código antes de leer. Funciona, pero depende de un hábito del modelo: si mañana agrega «Aquí tenés el JSON:», vuelve a romperse.

In [5]:
leidas_tras_limpiar = 0
for texto in respuestas_en_texto:
    limpio = texto.replace("```json", "").replace("```", "").strip()
    try:
        json.loads(limpio)
        leidas_tras_limpiar += 1
    except json.JSONDecodeError:
        pass

print(f"Tras quitar las marcas: {leidas_tras_limpiar} de {total}")

Tras quitar las marcas: 18 de 18


## 2. Un esquema que el modelo tiene que cumplir

En lugar de pedir con palabras, **declaramos la forma**. Con **Pydantic** escribimos una clase cuyos campos tienen nombre y tipo; `Literal` limita los valores posibles y `Field(ge=1, le=5)` pide un número entre 1 y 5. Ollama recibe el esquema en el parámetro `format` y restringe lo que el modelo puede escribir token por token. Después, Pydantic valida el resultado.

In [6]:
class AvisoClasificado(BaseModel):
    """Un aviso de vecino ya ordenado."""

    categoria: Literal["educación", "movilidad", "salud"]
    urgencia: int = Field(
        ge=1, le=5, description="1 es poco urgente, 5 muy urgente"
    )
    resumen: str = Field(description="Una oración con lo que dice el aviso")


esquema = AvisoClasificado.model_json_schema()
print(json.dumps(esquema, ensure_ascii=False, indent=2)[:700])

{
  "description": "Un aviso de vecino ya ordenado.",
  "properties": {
    "categoria": {
      "enum": [
        "educación",
        "movilidad",
        "salud"
      ],
      "title": "Categoria",
      "type": "string"
    },
    "urgencia": {
      "description": "1 es poco urgente, 5 muy urgente",
      "maximum": 5,
      "minimum": 1,
      "title": "Urgencia",
      "type": "integer"
    },
    "resumen": {
      "description": "Una oración con lo que dice el aviso",
      "title": "Resumen",
      "type": "string"
    }
  },
  "required": [
    "categoria",
    "urgencia",
    "resumen"
  ],
  "title": "AvisoClasificado",
  "type": "object"
}


In [7]:
def clasificar_con_esquema(aviso: str, instruccion: str) -> dict:
    """Pide un AvisoClasificado al modelo y lo valida.

    Args:
        aviso: Texto del aviso.
        instruccion: Texto del rol system.

    Returns:
        Diccionario con "objeto" (el AvisoClasificado o None si no cumplio
        el esquema) y "texto" (lo que escribio el modelo).
    """
    respuesta = ollama.chat(
        model=utils.MODELO_LOCAL,
        messages=[
            {"role": "system", "content": instruccion},
            {"role": "user", "content": f"Aviso: {aviso}"},
        ],
        format=esquema,
        options={"temperature": 0},
    )
    texto = respuesta.message.content
    try:
        objeto = AvisoClasificado.model_validate_json(texto)
    except ValidationError:
        objeto = None
    return {"objeto": objeto, "texto": texto}


ejemplo = clasificar_con_esquema(
    avisos_de_evaluacion[0]["texto"], "Clasificá el aviso de un vecino."
)
print(ejemplo["texto"])
print(type(ejemplo["objeto"]))

{ "categoria": "educación", "urgencia": 3, "resumen": "El curso de programación tendrá encuentros semanales." }
<class '__main__.AvisoClasificado'>


Lo medimos con dos instrucciones: una mínima y el contrato del cuaderno 02. Para cada una contamos cuántas respuestas cumplen el esquema y cuántas tienen la categoría correcta.

In [8]:
contrato = (
    "Sos un asistente del municipio. Clasificás avisos de vecinos en una "
    "categoría: educación, movilidad o salud. Resumí sin agregar datos "
    "que el aviso no diga."
)
instrucciones = {
    "mínima": "Clasificá el aviso de un vecino.",
    "con contrato": contrato,
}

resultados_de_esquema = []
for nombre, instruccion in instrucciones.items():
    cumplen_el_esquema = 0
    categoria_correcta = 0
    for registro in avisos_de_evaluacion:
        resultado = clasificar_con_esquema(registro["texto"], instruccion)
        if resultado["objeto"] is not None:
            cumplen_el_esquema += 1
            if resultado["objeto"].categoria == registro["etiqueta"]:
                categoria_correcta += 1
    resultados_de_esquema.append({
        "instrucción": nombre,
        "cumplen el esquema": cumplen_el_esquema,
        "categoría correcta": categoria_correcta,
        "total": len(avisos_de_evaluacion),
    })

utils.tabla(resultados_de_esquema)

,instrucción,cumplen el esquema,categoría correcta,total
0,mínima,18,14,18
1,con contrato,18,16,18


> **Para mirar.**
>
> - Con las dos instrucciones, las 18 respuestas cumplen el esquema: la forma la garantiza el mecanismo, no la redacción del pedido.
> - La categoría correcta sube de 14 a 16 con el contrato. El esquema no reemplaza una buena instrucción, y en ambos casos quedan errores: **válido no es correcto**.

## 3. Extraer datos de un aviso largo

Ahora el esquema tiene varios campos con huecos posibles: `str | None` significa «texto o nada». Le decimos al modelo que use `null` cuando un dato no aparece. El aviso no trae horarios ni precio: veamos qué escribe.

In [9]:
class AvisoDeActividad(BaseModel):
    """Datos de una actividad cultural anunciada en un texto."""

    nombre_del_evento: str
    lugar: str | None
    fecha_de_inicio: str | None
    fecha_de_cierre: str | None
    organizaciones: list[str]
    persona_de_contacto: str | None
    horario: str | None
    costo: str | None


instruccion_de_extraccion = (
    "Extraé los datos del aviso. Si un dato no aparece en el texto, usá null."
)
respuesta_larga = ollama.chat(
    model=utils.MODELO_LOCAL,
    messages=[
        {
            "role": "system",
            "content": instruccion_de_extraccion,
        },
        {"role": "user", "content": aviso_largo},
    ],
    format=AvisoDeActividad.model_json_schema(),
    options={"temperature": 0},
)
texto_de_actividad = respuesta_larga.message.content
actividad = AvisoDeActividad.model_validate_json(texto_de_actividad)
print(aviso_largo)

La Biblioteca Almafuerte de Rosario inauguró el ciclo "Invierno de lectura". Las actividades se realizan en el Centro Cultural del Puerto, con entrada libre. Ana Peralta, coordinadora del programa, presentó la agenda junto con la Cooperativa Orilla, que aportó los libros. Los encuentros comienzan el 3 de junio. El cierre será el 15 de agosto en el Anfiteatro Municipal de Santa Fe.


In [10]:
def aparece_en_el_texto(valor: str, texto: str) -> bool:
    """Indica si un valor extraido aparece literalmente en el texto.

    Args:
        valor: Texto extraido por el modelo.
        texto: Aviso original.

    Returns:
        True si el valor, sin distinguir mayusculas, esta en el texto.
    """
    return valor.lower() in texto.lower()


filas_de_actividad = []
for campo, valor in actividad.model_dump().items():
    if isinstance(valor, list):
        valores = valor
    elif valor is None:
        valores = []
    else:
        valores = [valor]
    esta_en_el_texto = "sin valor"
    if len(valores) > 0:
        esta_en_el_texto = "sí"
        for elemento in valores:
            if not aparece_en_el_texto(elemento, aviso_largo):
                esta_en_el_texto = "no"
    filas_de_actividad.append({
        "campo": campo,
        "valor extraído": valor,
        "aparece en el texto": esta_en_el_texto,
    })

utils.tabla(filas_de_actividad)

,campo,valor extraído,aparece en el texto
0,nombre_del_evento,Invierno de lectura,sí
1,lugar,Centro Cultural del Puerto,sí
2,fecha_de_inicio,3 de junio,sí
3,fecha_de_cierre,15 de agosto,sí
4,organizaciones,"[Biblioteca Almafuerte, Cooperativa Orilla]",sí
5,persona_de_contacto,Ana Peralta,sí
6,horario,entrada libre,sí
7,costo,None,sin valor


> **Para mirar.**
>
> - `horario` dice «entrada libre». Ese texto aparece en el aviso, y la verificación lo marca `sí`, pero «entrada libre» es el costo, no un horario. En cambio `costo` quedó vacío.
> - El aviso no trae ningún horario: lo correcto era `null` en `horario` y «entrada libre» en `costo`. El modelo respetó el tipo y puso el dato en el campo equivocado.
> - Comprobar que el valor esté en el texto detecta datos inventados, pero no detecta datos en el campo equivocado.

## 4. Un número no es una medición

El campo `urgencia` parece el más útil para ordenar el trabajo. Preguntamos dos cosas: si hay algún dato con que compararlo, y si el modelo da lo mismo cuando se repite el pedido.

In [11]:
urgencias_por_categoria = {"educación": [], "movilidad": [], "salud": []}
for registro in avisos_de_evaluacion:
    resultado = clasificar_con_esquema(registro["texto"], contrato)
    if resultado["objeto"] is not None:
        urgencias_por_categoria[registro["etiqueta"]].append(
            resultado["objeto"].urgencia
        )

filas_de_urgencia = []
for categoria, urgencias in urgencias_por_categoria.items():
    filas_de_urgencia.append({
        "categoría real": categoria,
        "urgencias asignadas": sorted(urgencias),
    })
utils.tabla(filas_de_urgencia)

,categoría real,urgencias asignadas
0,educación,"[1, 1, 1, 1, 3, 3]"
1,movilidad,"[1, 1, 3, 3, 3, 3]"
2,salud,"[1, 1, 3, 3, 3, 3]"


In [12]:
aviso_de_prueba = "Los usuarios reclaman un servicio nocturno de colectivos."

filas_de_repeticion = []
for semilla in (1, 2, 3):
    respuesta = ollama.chat(
        model=utils.MODELO_LOCAL,
        messages=[
            {"role": "system", "content": contrato},
            {"role": "user", "content": f"Aviso: {aviso_de_prueba}"},
        ],
        format=esquema,
        options={"temperature": 0.7, "seed": semilla},
    )
    objeto = AvisoClasificado.model_validate_json(respuesta.message.content)
    filas_de_repeticion.append({
        "semilla": semilla,
        "categoría": objeto.categoria,
        "urgencia": objeto.urgencia,
        "resumen": objeto.resumen,
    })
utils.tabla(filas_de_repeticion)

,semilla,categoría,urgencia,resumen
0,1,movilidad,3,Reclamo del servicio nocturno de colectivos
1,2,movilidad,3,El aviso se refiere a que los usuarios solicitan mejorar la disponibilidad de servicios nocturnos de transporte público.
2,3,movilidad,3,Los usuarios requieren mejorar la cobertura de servicios de transporte en horas nocturnas.


> **Para mirar.**
>
> - Las 18 urgencias valen 1 o 3: ninguna vale 2, 4 ni 5, y las tres categorías reparten los valores de forma parecida.
> - Con tres semillas, el mismo aviso recibió siempre `movilidad` y urgencia 3; el resumen cambió de redacción, no de contenido.
> - Que el modelo sea estable no prueba que acierte. En los datos no hay etiquetas de urgencia: no podemos decir si un 3 es correcto. Un número que nadie puede contrastar da una precisión aparente.

### Microlaboratorio 1 — Aceptar o rechazar una respuesta

**Tarea.** Escribir una función que lea una respuesta y devuelva el `AvisoClasificado` o `None`.  
**Producto.** `leer_respuesta`, comprobada con tres textos conocidos.  
**Antes de empezar.** Secciones 1 y 2.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `objeto` | el aviso validado | `AvisoClasificado.model_validate_json(texto)` |
| (2) `except` | atrapar solo las respuestas que no cumplen el esquema | `ValidationError` |

**Comprobación.** Un JSON correcto devuelve un objeto; el mismo JSON dentro de un bloque de código y uno con `urgencia` 9 devuelven `None`.

> **Ayuda 1.** `clasificar_con_esquema` hace exactamente esto dentro del `try`.  
> **Ayuda 2.** Una respuesta rechazada no tiene que cortar el cuaderno: la función devuelve `None`.

In [13]:
def leer_respuesta(texto: str) -> AvisoClasificado | None:
    """Convierte un texto JSON en un AvisoClasificado, o None.

    Args:
        texto: Respuesta del modelo.

    Returns:
        El aviso validado, o None si no cumple el esquema.
    """
    objeto = None  # COMPLETAR (1)
    return objeto


bueno = '{"categoria": "salud", "urgencia": 2, "resumen": "Hay turnos."}'
con_marcas = "```json\n" + bueno + "\n```"
fuera_de_rango = (
    '{"categoria": "salud", "urgencia": 9, "resumen": "Hay turnos."}'
)

utils.comprobar(
    leer_respuesta(bueno) is not None,
    "revisa (1): un JSON correcto tiene que devolver un AvisoClasificado",
)
rechazadas = [leer_respuesta(con_marcas), leer_respuesta(fuera_de_rango)]
utils.comprobar(
    rechazadas == [None, None],
    "revisa (2): los dos textos que no cumplen el esquema tienen que dar None",
)

Todavia no: revisa (1): un JSON correcto tiene que devolver un AvisoClasificado
Correcto.


## 5. El mismo esquema, en un modelo remoto

El esquema no depende del proveedor: Gemini acepta la misma clase de Pydantic como `response_schema` y devuelve el objeto ya armado en `parsed`. Es opcional y usa dos llamadas.

In [14]:
from google import genai
from google.genai import types

if utils.hay_clave("GEMINI_API_KEY"):
    cliente_gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
    respuesta_remota = cliente_gemini.models.generate_content(
        model=utils.MODELO_GEMINI,
        contents=f"Aviso: {aviso_de_prueba}",
        config=types.GenerateContentConfig(
            system_instruction=contrato,
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=AvisoClasificado,
        ),
    )
    print(respuesta_remota.parsed)
else:
    utils.avisar_sin_clave("GEMINI_API_KEY")

Sin GEMINI_API_KEY: se omite esta parte. Copia .env.example como .env, completa la clave y reinicia el kernel.


> **Para mirar.** Con clave, compará categoría y resumen con las filas de la sección 4. Que el objeto llegue armado no cambia la pregunta de fondo: la forma la garantiza el esquema, la verdad no.

### Microlaboratorio 2 — Qué campos se pueden automatizar

**Tarea.** Decidir, campo por campo, qué hacer con lo que devuelve el modelo.  
**Producto.** `decisiones` y `mi_decision`.  
**Antes de empezar.** Secciones 2, 3 y 4.

| Paso | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `decisiones` | para `categoria`, `urgencia` y `horario`, una de tres palabras | `"automatizar"`, `"revisar"` o `"descartar"` |
| Paso 2 | tu justificación en `mi_decision` | al menos 30 palabras que citen una evidencia de este cuaderno |

> **Ayuda 1.** Preguntate, para cada campo: ¿tengo datos con que medirlo? ¿da lo mismo si repito?  
> **Ayuda 2.** El caso de `horario` está en la sección 3 y el de `urgencia`, en la 4.

In [15]:
decisiones = {
    "categoria": "revisar",  # COMPLETAR (1)
    "urgencia": "revisar",
    "horario": "revisar",
}

valores_permitidos = ["automatizar", "revisar", "descartar"]
todas_validas = True
for campo, decision in decisiones.items():
    if decision not in valores_permitidos:
        todas_validas = False
utils.comprobar(
    todas_validas and decisiones["urgencia"] != "automatizar",
    "usa automatizar, revisar o descartar; mira la evidencia de urgencia",
)

mi_decision = ""
utils.comprobar(
    len(mi_decision.split()) >= 30,
    "escribi al menos 30 palabras en mi_decision",
)

Correcto.
Todavia no: escribi al menos 30 palabras en mi_decision


## Qué te llevás

- Pedir un JSON con palabras falla por detalles de formato. Un **esquema** (Pydantic más `format`) garantiza que la respuesta tenga la forma pedida y permite rechazar lo que no la cumple.
- **Válido no es verdadero**: un aviso puede llegar con la categoría equivocada, un campo vacío puede completarse con otro dato y un número como `urgencia` no tiene con qué medirse.
- Cada campo tiene su decisión: automatizar lo que se puede medir y es estable, revisar lo dudoso y descartar lo que no tiene evidencia.

**Sigue:** qué pasa cuando la respuesta depende de un dato que el modelo nunca vio (cuaderno 04).